# Bottlenecks in grievance disposal, July 2024 - June 25

Between July 2024 and June 2025, Odisha's Janasunani portal received roughly a little under seven lakh complaints and disposed around 70 per cent of these. For these disposed complaints, the average time to close was 63 days. The goal of this note is to decompose the total time taken into the stops at various offices a complaint traverses during its journey from receipt to closure, and identify the bottlenecks that cause most of the hold-up.

We find that most of this time is spent in the field. Of the 63 days, Stage 1, which runs from the receipt of a complaint to the selection of its route, takes 12 days (19 per cent). Stage 2, the action taken in the field for disposal, takes 45 days (71 per cent), and Stage 3, the review by the assigning authority, takes the remaining 6 days. The pattern holds across types of complaints: in each of the ten largest complaint types, Stage 2 accounts for most of the time taken. The automations proposed in Janasunani 2.0, which read, classify, route and triage complaints, all act on Stage 1. Even if they removed Stage 1 entirely, they would cut at most about a fifth of the time to disposal. This is a real efficiency gain, but it leaves most of the delay untouched.

We then look more closely at Stage 2, using complaints filed between July and December 2024. Most of the time in this stage is spent at the block level, with the Block Development Officer alone holding a complaint for around 28 days on average, more than half of the time spent in the field. The historical data, however, cannot tell us why. Offices that receive more complaints are not systematically slower, and offices that receive roughly the same number of complaints take very different amounts of time. Taken together, everything the portal records about a complaint, including the offices that handled it, its route, its type, its district and the month it was filed, explains only around 40 per cent of the variation in the time spent in Stage 2. Understanding the remaining 60 per cent requires visiting the field offices themselves and collecting data that the portal does not currently hold.


In [ ]:
# How many of the largest complaint types section 3.2 compares.
TOP_TYPES = 10

In [ ]:
import html

import polars as pl
from IPython.display import Markdown, display

from janasunani.analytics.bottlenecks import check_note, open_lake
from janasunani.analytics.exhibits import (
    draw_field_drivers, draw_field_levels, draw_office_volume, draw_stage_bar,
    draw_stage_stack, field_drivers, field_frame, field_layer_table, field_levels,
    field_roles, office_volume, stage_means, stage_rows, top_types,
)
from janasunani.analytics.figures import Report
from janasunani.analytics.journey import STAGES_THREE
from janasunani.analytics.sample import Sample

con = open_lake()
con.execute("SET enable_progress_bar = false")
pl.Config.set_fmt_str_lengths(120)
# Silent on pass: the CA&GR Analytics Note's published 2024-25 figures.
check_note(con)

# Tables are for a reader, not a debugger: no shape line, no dtype row, no
# quotes around text, blanks for nulls, thousands separators on counts. This
# also applies to tables the exhibits display, and survives export to Word.
def _cell(v):
    if v is None:
        return ""
    return f"{v:,}" if isinstance(v, int) else html.escape(str(v))


def _table_html(df):
    head = "".join(f"<th>{html.escape(c)}</th>" for c in df.columns)
    body = "".join("<tr>" + "".join(f"<td>{_cell(v)}</td>" for v in row) + "</tr>"
                   for row in df.rows())
    return f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"


def _table_md(df):
    rows = [list(df.columns), ["---"] * df.width,
            *[[_cell(v) for v in row] for row in df.rows()]]
    return "\n".join("| " + " | ".join(map(str, r)) + " |" for r in rows)


_fmt = get_ipython().display_formatter.formatters
_fmt["text/html"].for_type(pl.DataFrame, _table_html)
# The Word version is built from a Markdown export, which reads this one.
_fmt["text/markdown"].for_type(pl.DataFrame, _table_md)


STEPS = [
    ("F0", "Complaints filed in 2024-25", "TRUE"),
    ("F1", "Disposed", "is_disposed"),
    ("F2", "Disposed, with a usable closing date",
     "is_disposed AND days_to_close IS NOT NULL"),
]
FINAL = "Final sample: every office on the route identified, stage dates consistent"
FIGDIR = "../outputs/figures"
D2 = "Stage 2 days"
# The field-action part stands on complaints filed July to December 2024: a
# slow complaint filed late in the year is often still open.
H1 = "created_on < DATE '2025-01-01'"
def sample(name, where):
    s = Sample(con, ns=f"{name}_", base=f"year = '2024-25' AND {where}", steps=STEPS)
    s.install_route("F2", "F2")
    s.install_phases("F2", "F2")
    s.STEPS["P4"] = (FINAL, None)
    return s


def funnel(s):
    display(s.funnel_table(["F0", "F1", "F2", "P4"])
            .drop("Step").rename({"Grievances": "Complaints"}))


def decomposition(s, rep, label):
    means, total = stage_means(s, STAGES_THREE)
    draw_stage_bar(rep, means, total, f"Time to disposal by stage, {label}",
                   f"{s.FUNNEL['P4']:,} disposed complaints. Averages; the three "
                   f"stages add up to {total:.0f} days.", "02_stages", STAGES_THREE)


def by_type(s, rep, n):
    rows = stage_rows(s, [("All complaints", "TRUE")] + top_types(s, n), STAGES_THREE, floor=1)
    types = rows[1:]
    s1 = max(100 * r["Stage 1"] / r["total"] for _, r in types)
    s2 = min(100 * r["Stage 2"] / r["total"] for _, r in types)
    covered = sum(r["n"] for _, r in types)
    display(Markdown(
        f"**In each of the {n} largest complaint types, Stage 2 takes at least {s2:.0f}% "
        f"of the time to disposal, and Stage 1 at most {s1:.0f}%.** Together these types "
        f"are {100 * covered / s.FUNNEL['P4']:.0f}% of the sample."))
    draw_stage_stack(rep, rows, "Complaint type", f"Share of time to disposal by stage, {n} largest complaint types",
                     "Each bar is one complaint type, split into its stages; the number at the end "
                     "is its average days to disposal. Bracketed: complaints in the sample.",
                     "03_types", STAGES_THREE, count_label="Complaints", show_table=False, shares=True)


def field(s):
    frame = field_frame(s)
    display(Markdown(
        f"**{frame.height:,} complaints filed July to December 2024 reached Stage 2.** "
        f"On average they spent {frame['field_action'].mean():.0f} days in it."))
    return frame


def show_levels(s, frame, rep, label):
    levels, roles = field_levels(s, frame), field_roles(s, frame)
    draw_field_levels(rep, levels, roles, f"Stage 2 days by office level, {label}",
                      "Each day is counted at the office holding the complaint, "
                      "so the parts add up to the average.", "04_field_levels", days_label=D2)


def show_volume(s, frame, rep, label, floor=30):
    role = frame["field_role"].mode().sort()[0]
    offices = office_volume(s, frame, role, floor)
    display(Markdown(
        f"**There is no clear pattern between how many complaints a {role} office receives "
        "and how long it takes.** Offices receiving about the same number of complaints "
        "take very different times."))
    draw_office_volume(rep, offices, f"Complaints received and Stage 2 days by {role} office, {label}",
                       f"Every {role} office with at least {floor} disposed complaints. "
                       "Received counts every complaint that reached the office.",
                       "08_volume", days_label=D2)


def show_drivers(frame, rep, label):
    shares, full = field_drivers(frame)
    draw_field_drivers(rep, shares, full, f"Share of the variation in Stage 2 days, by factor, {label}",
                       f"{frame.height:,} complaints.", "05_field_drivers", days_label=D2)


## 1. Data and sample

Of the roughly 7 lakh complaints received, around 4.4 lakh complaints had consistent dates and the time taken at each office identified. Throughout this section, we use the sample. The details are listed precisely below in the table.

In [ ]:
state = sample("state", "TRUE")
assert state.FUNNEL["F0"] == 688_301, state.FUNNEL["F0"]   # the published total
funnel(state)

## 2. Three stages

| Stage | | Covers |
|---|---|---|
| **Stage 1** | Processing | From receipt to route selection |
| **Stage 2** | Field action | Action taken for disposal |
| **Stage 3** | Review | Review by the assigning authority |

Stage 3 is zero when a complaint is not sent back for review. The time from the last action to formal closure averages under a day and is left out.

## 3. Time to disposal by stage

The average total time taken to dispose a complaint in our sample can be broken up into the time taken in each of the stages enumerated above. We do this exercise for the full sample of complaints and for each of the ten largest complaint types in the data. We do a subcategory specific decomposition since different types of complaints could have different aggregate disposal times, and different decompositions.

### 3.1 Full sample
The average time for disposing the complaints in our full sample of roughly 4.4 lakh complaints is 63 days. This is then broken up into the time taken for each of the three stages: processing, field action, and review.

In [ ]:
rep = Report(con, [("Statewide", "TRUE")], FIGDIR, prefix="disp_state_")
decomposition(state, rep, "statewide")



Stage 2, the field action, takes 45 of the 63 days (71%). Stage 1, from receipt to route selection, takes 12 days (19%). Stage 3, review, takes 6 days (10%). The automations porposed in Janasunani 2.0 (reading the complaint, classifying it and choosing its route, flagging spam and duplicates) all act on Stage 1. If implemented perfectly so that the new system removed Stage 1 entirely without affecting quality, we would cut at most about a fifth of the time to disposal: 12 of 63 days. This is a real efficiency gain, but since most of the time taken is spent on the field, where the proposed automation does not reach, we need a better understanding of what the bottlenecks are and whether other forms of AI and automation can help ameliorate them.

### 3.2 By complaint type

Some types of complaints may need more work than others, so in order to check whether the stage decomposition holds for different types of complaints, we repeat the above analysis on each of the ten largest complaint types in our sample. Rural housing complaints alone make up a plurality (31%) of complaints in our sample, and thus represent the modal complaint; the smallest of the ten make up a little over one percent.

The total time to disposal differs considerably across these types, but the split between the stages does not. In every one of them, Stage 2 takes up most of the time and Stage 1 a fifth or so, much like the overall share.

In [ ]:
by_type(state, rep, TOP_TYPES)

### 3.3 Summary

*Draft points, to be rewritten as prose:*


- **Most of the time is in Stage 2.** About 70% of the time to disposal statewide, and most of it in each of the ten largest complaint types.
- **Stage 1 is about a fifth** of the time statewide, and at most about a quarter in any of the ten largest types.
- **What Janasunani 2.0 can do.** Its automations act on Stage 1, so they can cut at most about a fifth of the time to disposal. Significant, but not the whole problem.
- **What remains.** Reducing the rest means understanding what happens in the field, which is the subject of the next part.

<div style="page-break-before: always"></div>

## 4. Stage 2 in detail

*Draft points, to be rewritten as prose:*


- We know exactly where the time goes: Stage 2, and within it the block office.
- The record does not tell us why. Three pieces of evidence:
  1. **Where Stage 2 time is spent** (4.1), by office level and type of office.
  2. **Whether busier offices are slower** (4.2). If volume drove the time, more staff would be the answer.
  3. **How much of the variation the record can explain at all** (4.3).
- Together they say the field needs to be seen first-hand, and that more data needs to be collected.

This part is statewide and uses complaints filed July to December 2024, because slow complaints filed later are often still open. Each day of Stage 2 is counted at the office holding the complaint. The factors in the third piece of evidence are defined beside it; where two overlap, such as an office and its district, the shared part is split evenly.

In [ ]:
state_frame = field(st := sample("state_h1", H1))

### 4.1 Stage 2 days by office level

In [ ]:
rep = Report(con, [("Statewide", "TRUE")], FIGDIR, prefix="disp_state_")
show_levels(st, state_frame, rep, "statewide")

*Draft points, to be rewritten as prose:*


- Statewide, about two thirds of Stage 2 time (32 of 51 days) is spent at the block or field office. The BDO alone holds the complaint for 28.5 days on average, over half of Stage 2.
- This tells us where the time accumulates, but not why: the record shows when a complaint arrived and left, not what was done in between.

### 4.2 Complaints received and Stage 2 days

In [ ]:
show_volume(st, state_frame, rep, "statewide")

*Draft points, to be rewritten as prose:*

- If the BDO is slow because it has too many complaints, busier offices should take longer. There is no clear pattern: busy and quiet offices take similar times.
- At any given volume the variation is enormous: offices receiving about the same number of complaints range from under 20 days to over 100.
- So volume is not the explanation. Something about how particular offices work, or the cases they get, is. That needs someone to go and look.

### 4.3 Share of the variation in Stage 2 days explained by the record

In [ ]:
display(field_layer_table())

In [ ]:
show_drivers(state_frame, rep, "statewide")

*Draft points, to be rewritten as prose:*


- Everything the record holds about a complaint (which office handled it, who sent it, the route, complaint type, district, how it entered, month) explains about 40% of the variation in Stage 2 days statewide.
- The field office and the office that sent the complaint explain the most, about a tenth each. Complaint type explains about 4%.
- The remaining ~60% is not explained by anything recorded. The answer is in what the record does not capture.

### 4.4 Limitations of the historical data

*Draft points, to be rewritten as prose:*


- **We know where the time is.** Stage 2, and within it the BDO's office.
- **It is not volume.** There is no clear pattern between an office's volume and its time.
- **Most of the variation is unexplained.** The recorded factors explain about 40% of it; offices of similar volume take very different times.
- **Next steps.** [Field visits to a set of fast and slow offices of similar volume; record why a complaint waits (enquiry, the citizen, another department); collect staffing and workload data the portal does not hold.]

In [ ]:
con.close()